In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.datasets import make_regression
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import train_test_split

from deepordinal.tf import OrdinalOutput, ordinal_loss

In [ ]:
x, y = make_regression(n_samples=10000, n_features=16, n_informative=12)

qt = QuantileTransformer()
y = qt.fit_transform(y.reshape(-1, 1))[:, 0]
y = np.floor(y * 4).astype(np.int32)  # 4 ordinal classes: 0, 1, 2, 3
y = np.clip(y, 0, 3)

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.25)
print("Class distribution:", np.bincount(y_train))

In [ ]:
dense1 = tf.keras.layers.Dense(16, activation='relu')
dense2 = tf.keras.layers.Dense(16, activation='relu')
ordinal = OrdinalOutput(output_dim=4)

# Build layers by calling once
dummy = dense2(dense1(x_train[:1].astype(np.float32)))
_ = ordinal(dummy)

optimizer = tf.keras.optimizers.Adam()
print("Model built.")

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (x_train.astype(np.float32), y_train)
).shuffle(len(x_train)).batch(16)

val_ds = tf.data.Dataset.from_tensor_slices(
    (x_val.astype(np.float32), y_val)
).batch(16)

all_params = dense1.trainable_variables + dense2.trainable_variables + ordinal.trainable_variables
epochs = 50
train_losses, val_losses = [], []

for epoch in range(epochs):
    epoch_loss = []
    for xb, yb in train_ds:
        with tf.GradientTape() as tape:
            h = dense2(dense1(xb))
            logits = tf.matmul(h, ordinal.kernel) + ordinal.bias
            thresholds = ordinal.interior_thresholds[0]
            loss = ordinal_loss(logits, yb, thresholds)
        grads = tape.gradient(loss, all_params)
        optimizer.apply_gradients(zip(grads, all_params))
        epoch_loss.append(loss.numpy())
    train_losses.append(np.mean(epoch_loss))

    v_loss = []
    for xb, yb in val_ds:
        h = dense2(dense1(xb))
        logits = tf.matmul(h, ordinal.kernel) + ordinal.bias
        thresholds = ordinal.interior_thresholds[0]
        v_loss.append(ordinal_loss(logits, yb, thresholds).numpy())
    val_losses.append(np.mean(v_loss))

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}  train_loss={train_losses[-1]:.4f}  val_loss={val_losses[-1]:.4f}")

In [ ]:
x_val_t = tf.constant(x_val.astype(np.float32))
probs = ordinal(dense2(dense1(x_val_t)))
preds = tf.argmax(probs, axis=1).numpy()
accuracy = np.mean(preds == y_val)
print(f"Validation accuracy: {accuracy:.4f}")

In [ ]:
plt.plot(train_losses, '--', label='train_loss')
plt.plot(val_losses, label='val_loss')
plt.title('Training and validation loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend()
plt.show()